# Lesson 6 — Teleportation and Superdense Coding

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 6. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- Quantum teleportation: protocol and circuit
- Classical corrections with `if_test` (Qiskit 1.x)
- Verifying teleportation with shot-based tests
- The deferred measurement principle
- Superdense coding: encoding and decoding
- The no-cloning theorem and faster-than-light argument

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. The Teleportation Protocol

Alice holds an unknown qubit $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ on $q_0$.
She and Bob pre-share the Bell pair $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ on $q_1$ (Alice) and $q_2$ (Bob).

**Alice's operations:**
1. CNOT: $q_0$ controls $q_1$
2. Hadamard on $q_0$
3. Measure $q_0 \to c_0$, $q_1 \to c_1$

After Alice's operations, the three-qubit state grouped by outcome $(c_0, c_1)$ is:

$$\frac{1}{2}\Bigl[
  |0,0\rangle(\alpha|0\rangle + \beta|1\rangle)
+ |0,1\rangle(\alpha|1\rangle + \beta|0\rangle)
+ |1,0\rangle(\alpha|0\rangle - \beta|1\rangle)
+ |1,1\rangle(\alpha|1\rangle - \beta|0\rangle)
\Bigr]$$

**Bob's corrections:**
- If $c_1 = 1$: apply $X$ to $q_2$
- If $c_0 = 1$: apply $Z$ to $q_2$

In every case, $q_2$ ends up holding exactly $|\psi\rangle$.

| $c_0$ | $c_1$ | State of $q_2$ | Bob applies |
|:--:|:--:|:--|:--|
| 0 | 0 | $\alpha\|0\rangle + \beta\|1\rangle$ | nothing |
| 0 | 1 | $\alpha\|1\rangle + \beta\|0\rangle$ | $X$ |
| 1 | 0 | $\alpha\|0\rangle - \beta\|1\rangle$ | $Z$ |
| 1 | 1 | $\alpha\|1\rangle - \beta\|0\rangle$ | $X$ then $Z$ |

## 2. Building the Circuit

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import numpy as np

sim = AerSimulator()

def teleportation_circuit(state_vec):
    """
    Teleportation circuit. state_vec is [alpha, beta].

    Classical register layout (single ClassicalRegister(3)):
      c[0]: q0 measurement  -> controls Z on q2
      c[1]: q1 measurement  -> controls X on q2
      c[2]: q2 output measurement
    """
    qr = QuantumRegister(3, 'q')
    cr = ClassicalRegister(3, 'c')
    qc = QuantumCircuit(qr, cr)

    # Prepare the unknown state on q0
    qc.initialize(state_vec, qr[0])
    qc.barrier()

    # Bell pair on q1 (Alice) and q2 (Bob)
    qc.h(qr[1])
    qc.cx(qr[1], qr[2])
    qc.barrier()

    # Alice's Bell measurement operations
    qc.cx(qr[0], qr[1])
    qc.h(qr[0])
    qc.barrier()

    # Alice measures
    qc.measure(qr[0], cr[0])   # c[0] = q0 result
    qc.measure(qr[1], cr[1])   # c[1] = q1 result
    qc.barrier()

    # Bob's corrections via if_test (Qiskit 1.x API)
    with qc.if_test((cr[1], 1)):
        qc.x(qr[2])
    with qc.if_test((cr[0], 1)):
        qc.z(qr[2])

    # Measure Bob's output
    qc.measure(qr[2], cr[2])
    return qc

qc = teleportation_circuit([1, 0])
qc.draw('mpl', fold=60)

The `if_test` context manager is Qiskit 1.x's replacement for the removed `c_if` method. The construct `with qc.if_test((cr[i], value)):` applies the enclosed gates only when classical bit `cr[i]` equals `value`.

## 3. Verifying the Protocol

For states with known measurement probabilities, the output distribution of $q_2$ should match the input.

The bitstring from `get_counts()` for `ClassicalRegister(3)` is `c[2]c[1]c[0]` in big-endian order. The leftmost character is `c[2]`, which is Bob's output qubit.

In [ ]:
def run_teleport(state_vec, label, shots=4096):
    qc = teleportation_circuit(state_vec)
    counts = sim.run(qc, shots=shots).result().get_counts()
    # Bitstring is 'c[2]c[1]c[0]'; c[2] is the leftmost character
    out_0 = sum(v for k, v in counts.items() if k[0] == '0')
    out_1 = sum(v for k, v in counts.items() if k[0] == '1')
    print(f"Teleporting {label:8s}: P(q2=0) = {out_0/shots:.3f}, P(q2=1) = {out_1/shots:.3f}")

run_teleport([1, 0],                        '|0⟩')   # expect P(0)=1.0
run_teleport([0, 1],                        '|1⟩')   # expect P(1)=1.0
run_teleport([1/np.sqrt(2), 1/np.sqrt(2)],  '|+⟩')   # expect 0.5, 0.5
run_teleport([1/np.sqrt(2), -1/np.sqrt(2)], '|−⟩')   # expect 0.5, 0.5

## 4. The Deferred Measurement Principle

Any mid-circuit measurement followed by classical conditioning can be replaced by a controlled quantum gate, with the measurement deferred to the end. The two circuits produce identical output statistics.

In the teleportation circuit:
- "measure $q_1$; if 1, apply $X$ to $q_2$" becomes $\text{CX}(q_1, q_2)$
- "measure $q_0$; if 1, apply $Z$ to $q_2$" becomes $\text{CZ}(q_0, q_2)$

The unitary version has no measurements inside the circuit, so `Statevector` and `partial_trace` can verify the output state exactly.

In [ ]:
from qiskit.quantum_info import Statevector, partial_trace, DensityMatrix, state_fidelity

def teleport_deferred(state_vec):
    """
    Unitary teleportation using the deferred measurement principle.
    No mid-circuit measurements. Corrections are coherent controlled gates.
    After the circuit, q2 holds exactly |psi>.
    """
    qc = QuantumCircuit(3)
    qc.initialize(state_vec, 0)  # q0: Alice's message
    # Bell pair on q1 and q2
    qc.h(1)
    qc.cx(1, 2)
    # Alice's operations
    qc.cx(0, 1)
    qc.h(0)
    # Deferred corrections
    qc.cx(1, 2)    # X on q2 controlled by q1
    qc.cz(0, 2)    # Z on q2 controlled by q0
    return qc

test_states = {
    '|0⟩':  np.array([1, 0]),
    '|1⟩':  np.array([0, 1]),
    '|+⟩':  np.array([1, 1]) / np.sqrt(2),
    '|+i⟩': np.array([1, 1j]) / np.sqrt(2),
    '|T⟩':  np.array([1, np.exp(1j * np.pi / 4)]) / np.sqrt(2),
}

print("Teleportation fidelities (deferred measurement version):")
print()
for name, psi in test_states.items():
    qc = teleport_deferred(psi)
    sv = Statevector(qc)
    # partial_trace(sv, [0, 1]) traces out q0 and q1, leaving q2
    rho_q2 = partial_trace(sv, [0, 1])
    fidelity = state_fidelity(rho_q2, DensityMatrix(psi))
    print(f"  {name:6s}: fidelity = {np.real(fidelity):.6f}")

In [ ]:
# Compare circuits side by side
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
teleportation_circuit([1, 0]).draw('mpl', ax=axes[0], fold=40)
axes[0].set_title('Standard circuit (mid-circuit measurement + if_test)')
teleport_deferred(np.array([1, 0])).draw('mpl', ax=axes[1])
axes[1].set_title('Deferred measurement circuit (fully unitary)')
plt.tight_layout()
plt.show()

## 5. Superdense Coding

Superdense coding is the complementary protocol. Alice and Bob pre-share $|\Phi^+\rangle$. Alice encodes two classical bits by applying local gates to her qubit, sends the qubit to Bob, and Bob performs a Bell measurement to decode.

**Encoding table:**

| $(b_0, b_1)$ | Alice applies | Bell state |
|:--:|:--:|:--|
| 0, 0 | nothing | $|\Phi^+\rangle = (|00\rangle+|11\rangle)/\sqrt{2}$ |
| 1, 0 | $X$ | $|\Psi^+\rangle = (|10\rangle+|01\rangle)/\sqrt{2}$ |
| 0, 1 | $Z$ | $|\Phi^-\rangle = (|00\rangle-|11\rangle)/\sqrt{2}$ |
| 1, 1 | $X$ then $Z$ | $|\Psi^-\rangle = (|01\rangle-|10\rangle)/\sqrt{2}$ |

**Decoding:** CNOT($q_0 \to q_1$), then H($q_0$), then measure both.

In [ ]:
def superdense_encode(b0, b1):
    """
    Prepare Bell pair and encode two classical bits.
    b0 controls the X gate; b1 controls the Z gate.
    """
    qc = QuantumCircuit(2, 2)
    # Create Bell pair |Phi+>
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    # Alice's encoding on q0
    if b0:
        qc.x(0)
    if b1:
        qc.z(0)
    return qc

def superdense_decode(qc):
    """
    Bob's Bell measurement. Appends decoding in place.
    After decoding: c[0] = b1, c[1] = b0.
    """
    qc.barrier()
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)   # q0 -> c[0] = b1
    qc.measure(1, 1)   # q1 -> c[1] = b0
    return qc

# Show the circuit for message (1, 0)
qc_ex = superdense_encode(1, 0)
superdense_decode(qc_ex)
qc_ex.draw('mpl')

In [ ]:
# Round-trip test for all four messages
print("Superdense coding round-trip:")
print()
for b0, b1 in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    qc = superdense_encode(b0, b1)
    superdense_decode(qc)
    counts = sim.run(qc, shots=1024).result().get_counts()
    # Bitstring is 'c[1]c[0]' (big-endian): c[1]=b0, c[0]=b1
    decoded = max(counts, key=counts.get)
    b0_dec = int(decoded[0])   # c[1] = b0
    b1_dec = int(decoded[1])   # c[0] = b1
    ok = (b0_dec, b1_dec) == (b0, b1)
    print(f"  Sent ({b0},{b1}), received ({b0_dec},{b1_dec})  {'OK' if ok else 'FAIL'}")

### Comparing the two protocols

| | Teleportation | Superdense coding |
|:--|:--|:--|
| What is transferred | 1 qubit state | 2 classical bits |
| Channel used | 1 classical channel (2 bits) | 1 quantum channel (1 qubit) |
| Pre-shared resource | 1 Bell pair | 1 Bell pair |
| Alice's operation | Bell measurement | Single-qubit gates |
| Bob's operation | Pauli corrections | Bell measurement |

## 6. The No-Cloning Theorem

No unitary $U$ satisfies $U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$ for all $|\psi\rangle$.

**Proof by linearity:** If $U|0\rangle|0\rangle = |00\rangle$ and $U|1\rangle|0\rangle = |11\rangle$, then for $|{+}\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$:

$$U|{+}\rangle|0\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}} = |\Phi^+\rangle$$

But cloning requires $U|{+}\rangle|0\rangle = |{+}\rangle|{+}\rangle = (|00\rangle + |01\rangle + |10\rangle + |11\rangle)/2$. These are different states. The contradiction proves no such $U$ exists.

Teleportation is consistent: after Alice's Bell measurement, her qubits are projected to a product state and the original superposition is destroyed. At the moment Bob's correction completes, exactly one copy of $|\psi\rangle$ exists.

In [ ]:
# Bob's qubit is maximally mixed before he receives the classical bits

def teleport_no_correction(state_vec):
    """Teleportation circuit without Bob's corrections."""
    qc = QuantumCircuit(3)
    qc.initialize(state_vec, 0)
    qc.h(1); qc.cx(1, 2)
    qc.cx(0, 1); qc.h(0)
    # Corrections deliberately omitted
    return qc

print("q2 density matrix WITHOUT corrections (Bob has not received Alice's bits):")
print()
for name, psi in list(test_states.items())[:3]:
    qc = teleport_no_correction(psi)
    sv = Statevector(qc)
    rho_q2 = partial_trace(sv, [0, 1])
    print(f"  Input {name}:")
    print(np.round(rho_q2.data, 4))
    print()

print("Every input gives I/2: Bob cannot infer anything about |ψ⟩ without the classical bits.")

## 7. Exercises

### Exercise 1: Teleporting a phase state

Teleport $|{-}\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$.

1. Use `run_teleport` to verify that Bob's output shows 50/50 statistics.
2. Use `teleport_deferred` with `partial_trace` to confirm the fidelity is exactly 1.
3. What is the expected density matrix of $q_2$ after a perfect teleportation of $|{-}\rangle$? Compute it and compare to `rho_q2`.

In [ ]:
# Exercise 1: your solution here
psi_minus = np.array([1, -1]) / np.sqrt(2)

# 1. Shot-based test
# TODO

# 2. Deferred measurement fidelity
# TODO

# 3. Expected density matrix
# TODO

### Exercise 2: Teleportation with a different Bell pair

Change the shared Bell pair from $|\Phi^+\rangle$ to $|\Phi^-\rangle = (|00\rangle - |11\rangle)/\sqrt{2}$ by inserting a $Z$ gate on $q_1$ after the CNOT in the Bell pair preparation.

1. Rederive the correction table. Which corrections does Bob need for each $(c_0, c_1)$ outcome?
2. Write a modified `teleportation_circuit_phi_minus` that applies the right corrections for this Bell pair.
3. Verify it with `run_teleport` for $|0\rangle$, $|1\rangle$, and $|{+}\rangle$.

In [ ]:
# Exercise 2: your solution here
def teleportation_circuit_phi_minus(state_vec):
    qr = QuantumRegister(3, 'q')
    cr = ClassicalRegister(3, 'c')
    qc = QuantumCircuit(qr, cr)
    qc.initialize(state_vec, qr[0])
    qc.barrier()
    # Bell pair |Phi->
    qc.h(qr[1])
    qc.cx(qr[1], qr[2])
    qc.z(qr[1])   # flip sign to get |Phi->
    qc.barrier()
    qc.cx(qr[0], qr[1])
    qc.h(qr[0])
    qc.barrier()
    qc.measure(qr[0], cr[0])
    qc.measure(qr[1], cr[1])
    qc.barrier()
    # TODO: add correct Bob corrections for |Phi->
    qc.measure(qr[2], cr[2])
    return qc

# TODO: test with run_teleport

### Exercise 3: Gate order in superdense coding

For $(b_0, b_1) = (1, 1)$, the encoding applies $X$ then $Z$ to Alice's qubit.

1. What Bell state results if Alice instead applies $Z$ then $X$ (reversed order)?
2. Use `Statevector` to compute both and compare their density matrices.
3. Run the reversed-order version through `superdense_decode` with 1024 shots. Does Bob recover $(1, 1)$ correctly? Explain why (hint: global phase).

In [ ]:
# Exercise 3: your solution here
from qiskit.quantum_info import Statevector

# XZ encoding
qc_xz = QuantumCircuit(2)
qc_xz.h(0); qc_xz.cx(0, 1)
qc_xz.x(0); qc_xz.z(0)
sv_xz = Statevector.from_label('00').evolve(qc_xz)
print("XZ encoding:", sv_xz.data.round(3))

# ZX encoding
qc_zx = QuantumCircuit(2)
qc_zx.h(0); qc_zx.cx(0, 1)
# TODO: apply Z then X instead
# sv_zx = ...

# TODO: run through superdense_decode and check counts

### Exercise 4: Multi-message superdense transmission

Encode the ASCII character `'Q'` (decimal 81, binary `01010001`) as four 2-bit chunks using four separate Bell pairs.

1. Split 81 into four 2-bit chunks: `(b0, b1)` pairs extracted from the binary representation.
2. Run four separate superdense coding round-trips and collect the decoded bits.
3. Reconstruct the 8-bit value and verify it equals 81.
4. What is the ASCII character for decimal 107 (`'k'`)? Transmit it the same way.

In [ ]:
# Exercise 4: your solution here
def encode_char(char):
    """Split ASCII character into four 2-bit chunks and transmit via superdense coding."""
    val = ord(char)
    bits = [(val >> i) & 1 for i in range(8)]  # 8 bits LSB first
    # TODO: pair into 4 chunks of (b0, b1) and transmit each
    # TODO: reconstruct and verify
    pass

encode_char('Q')

## Summary

In this lesson you:

- Built a quantum teleportation circuit using `if_test` for classical conditioning and verified that Bob's output matches the input for computational basis states and superpositions.
- Applied the deferred measurement principle by replacing mid-circuit measurements with `cx` and `cz` gates, then confirmed fidelity = 1 for five different input states using `partial_trace` and `state_fidelity`.
- Implemented superdense coding with `superdense_encode` and `superdense_decode` and verified all four messages in a round-trip test.
- Proved the no-cloning theorem from linearity and demonstrated that Bob's qubit is $I/2$ before receiving Alice's classical bits, confirming the protocol cannot signal faster than light.

**Lesson 7** introduces the oracle model of computation and the first quantum algorithms with a provable advantage: Deutsch-Jozsa answers a global question about a function with a single query, and Bernstein-Vazirani recovers a hidden bit string in one query where a classical computer needs $n$.